<a href="https://colab.research.google.com/github/rajputans/rajputans/blob/main/Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('train.txt',sep = ';',header = None,names = ['text','emotion'])

In [5]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [6]:
df.isnull().sum()

,0
text,0
emotion,0


In [7]:
unique_emotions = df['emotion'].unique()
emotion_numbers = {}
i = 0
for emotion in unique_emotions:
    emotion_numbers[emotion] = i
    i += 1
df['emotion'] = df['emotion'].map(emotion_numbers)

In [8]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


In [9]:
df['text'] = df['text'].apply(lambda x: x.lower())

In [10]:
import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))
df['text'] = df['text'].apply(remove_punctuation)

In [11]:
def remove_numbers(txt):
  new = ""
  for i in txt:
    if not i.isdigit():
      new += i
  return new
df['text'] = df['text'].apply(remove_numbers)

In [12]:
def remove_emojis(txt):
  new = ""
  for i in txt:
    if i.isascii():
      new += i
  return new
df['text'] = df['text'].apply(remove_emojis)

In [13]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [14]:
stop_words = set(stopwords.words('english'))

In [15]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [16]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)
  return ' '.join(cleaned)
df['text'] = df['text'].apply(remove)

In [17]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [18]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer

In [34]:
bow_vectorizer = CountVectorizer()

In [50]:
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [51]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [52]:
nb_model = MultinomialNB()

In [53]:
nb_model.fit(X_train_bow,y_train)

MultinomialNB()

In [54]:
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test,pred_bow))

0.768125


In [55]:
pred_bow

array([0, 5, 0, ..., 5, 5, 0])

In [56]:
y_test

,emotion
8756,0
4660,5
6095,0
304,5
8241,0
...,...
15578,5
5746,5
6395,5
7624,5


In [60]:
tfidf_vectorizer = TfidfVectorizer()

# Train data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Test data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [62]:
nb2_model = MultinomialNB()

nb2_model.fit(X_train_tfidf, y_train)

MultinomialNB()

In [64]:
pred = nb2_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.6609375


In [65]:
from sklearn.linear_model import LogisticRegression

In [68]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [69]:
log_pred = logistic_model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, log_pred))

Accuracy: 0.8628125
